# Step 1 — Prepare Data

## 1. Objective

Import checksum-verified, demultiplexed FASTQ files from a QIIME 2 manifest into a typed artifact and produce a read-count/quality summary for downstream review.

## 2. Scientific background

A QIIME 2 artifact couples data with a semantic type and provenance. Manifest import is appropriate when each sample already has demultiplexed FASTQ reads.

## 3. Methodological rationale

Read layout selects the semantic type and Phred33 V2 manifest format explicitly. This step does not trim primers, filter samples, denoise reads, or silently reuse an older artifact; those operations require their own declared contracts and parameters.

## 4. Inputs

The effective YAML supplies `inputs.metadata_file`, `inputs.manifest_file`, and `sequencing.read_layout`. Preflight requires exact metadata/manifest sample correspondence, readable literal absolute FASTQ paths, and valid pairing before this notebook runs.

## 5. Parameters

Papermill replaces the following single parameter cell from the effective run snapshot.

In [ ]:
experiment_name = "example-study"
base_dir = "."
inputs = {"metadata_file": "metadata.tsv", "manifest_file": "manifest.tsv"}
sequencing = {"read_layout": "single-end"}
resources = {"threads": 1}
pipeline = {"steps": ["prepare-data"]}
prepare_data = {"phred_offset": 33, "quality_plot_reads": 10000}


## 6. Computational provenance

In [ ]:
import json
import os
import platform
from datetime import UTC, datetime
from pathlib import Path

import rachis.plugins.demux.actions as demux_actions
from qiime2 import Artifact, Metadata
from qiime2 import __version__ as qiime2_version

RUN_ID = os.environ["AMPLICONFLOW_RUN_ID"]
RUN_DIR = Path(os.environ["AMPLICONFLOW_RUN_DIR"]).resolve()
PLAN_FILE = Path(os.environ["AMPLICONFLOW_PLAN_FILE"]).resolve()
print({"run_id": RUN_ID, "experiment": experiment_name, "python": platform.python_version(), "qiime2": qiime2_version})


## 7. Analysis

The method-defining QIIME 2 calls remain visible: validate metadata, import the manifest with its declared semantic/view types, and summarize the demultiplexed reads.

In [ ]:
manifest_path = Path(inputs["manifest_file"]).resolve(strict=True)
metadata_path = Path(inputs["metadata_file"]).resolve(strict=True)
layout = sequencing["read_layout"]
phred_offset = prepare_data.get("phred_offset", 33)
quality_plot_reads = prepare_data.get("quality_plot_reads", 10000)
if layout not in {"single-end", "paired-end"}:
    raise ValueError("Unsupported read layout")
if phred_offset != 33:
    raise ValueError("Prepare Data currently supports only Phred33 manifests")
if not isinstance(quality_plot_reads, int) or isinstance(quality_plot_reads, bool) or quality_plot_reads < 1:
    raise ValueError("prepare_data.quality_plot_reads must be a positive integer")

semantic_type = ("SampleData[SequencesWithQuality]" if layout == "single-end" else "SampleData[PairedEndSequencesWithQuality]")
view_type = ("SingleEndFastqManifestPhred33V2" if layout == "single-end" else "PairedEndFastqManifestPhred33V2")
metadata = Metadata.load(metadata_path)
print({"manifest": str(manifest_path), "metadata_samples": len(metadata.ids), "semantic_type": semantic_type, "view_type": view_type})


In [ ]:
step_artifacts = RUN_DIR / "artifacts" / "prepare-data"
step_figures = RUN_DIR / "figures" / "prepare-data"
step_reports = RUN_DIR / "reports" / "contributions"
step_figures.mkdir(parents=True, exist_ok=False)
step_reports.mkdir(parents=True, exist_ok=True)
artifact_path = step_artifacts / "demultiplexed_sequences.qza"
visualization_path = step_figures / "demultiplexed_sequences.qzv"

demultiplexed_sequences = Artifact.import_data(semantic_type, manifest_path, view_type=view_type)
demultiplexed_sequences.save(artifact_path)
summary, = demux_actions.summarize(data=demultiplexed_sequences, n=quality_plot_reads)
summary.save(visualization_path)


## 8. Results and quality-control assessment

Acceptance requires a loadable artifact with the planned semantic type and a demux summary visualization. Per-sample read counts and positional quality plots guide the later DADA2 decision; they are not interpreted automatically here.

## 9. Interpretation

Successful import establishes that the manifest representation is compatible with QIIME 2. It does not establish adequate read quality or justify truncation parameters.

In [ ]:
plan = json.loads(PLAN_FILE.read_text(encoding="utf-8"))
planned_step = next(step for step in plan["steps"] if step["id"] == "prepare-data")
contribution = {
    "schema_version": 1,
    "step": "prepare-data",
    "run_id": RUN_ID,
    "created_at": datetime.now(UTC).isoformat(),
    "objective": "Import demultiplexed FASTQ reads into a typed QIIME 2 artifact.",
    "methods": {"semantic_type": semantic_type, "view_type": view_type, "quality_plot_reads": quality_plot_reads},
    "outputs": {"demultiplexed_sequences": planned_step["outputs"]["demultiplexed_sequences"]["path"], "quality_summary": str(visualization_path.relative_to(RUN_DIR))},
    "limitations": ["Import success does not establish read quality or select DADA2 trimming parameters."],
    "references": ["https://doi.org/10.1038/s41587-019-0209-9"],
}
contribution_path = step_reports / "prepare-data.json"
with contribution_path.open("x", encoding="utf-8") as handle:
    json.dump(contribution, handle, indent=2, sort_keys=True)
    handle.write("\n")
print({"artifact": str(artifact_path), "visualization": str(visualization_path), "report_contribution": str(contribution_path)})


## 10. Outputs

- `artifacts/prepare-data/demultiplexed_sequences.qza`
- `figures/prepare-data/demultiplexed_sequences.qzv`
- `reports/contributions/prepare-data.json`

## 11. Limitations

Only Phred33 manifest import is supported. Primer trimming from the legacy notebook is intentionally not performed here. The summary samples reads randomly and therefore its plotted subsample is not a byte-stable result.

## 12. References

Bolyen E, et al. Reproducible, interactive, scalable and extensible microbiome data science using QIIME 2. *Nature Biotechnology*. 2019;37:852–857. https://doi.org/10.1038/s41587-019-0209-9